# MuJoCo vs Isaac Sim: A Practical Benchmark

**Most engine comparisons measure only speed, which is easy to win by being wrong. This one measures both, on hardware you can actually get for free.**

---

> Part of an open series on running **NVIDIA Isaac Sim on free GPUs**.
> Isaac Sim is free software (Apache 2.0) — only compute ever costs money,
> and this series is about not paying for that either.
>
> If this is useful, an upvote helps other people find it. Questions in the
> comments get answered.

---

## The question

If you are picking a physics engine for robot learning, you will find plenty
of benchmarks claiming X is N times faster than Y. Almost all of them share a
flaw: **speed alone is a meaningless metric for a simulator.**

Any integrator can be made arbitrarily fast by taking larger timesteps and
accepting more error. A benchmark that reports steps/second without reporting
accuracy is measuring how willing an engine is to be wrong quickly.

So this notebook measures two things:

1. **Throughput** — steps/second on a contact-rich scene, scaling from 1 to
   512 bodies.
2. **Accuracy** — energy drift of an undamped pendulum, which has an exact
   conserved quantity. Any drift is pure integrator error.

The second one is where it gets interesting.

### The engines

| | MuJoCo | Isaac Sim |
|---|---|---|
| Solver | Convex, soft-constraint | PhysX 5 TGS |
| Hardware | CPU (MJX adds GPU) | NVIDIA GPU required |
| Runs on Apple Silicon | ✅ | ❌ |
| Runs on free Kaggle T4 | ✅ | ✅ (see notebook 1) |
| Photorealistic rendering | ✗ | ✅ RTX |
| License | Apache 2.0 | Apache 2.0 |

The hardware row is the practical headline. MuJoCo runs anywhere — the
results below were produced on an ARM laptop with no discrete GPU at all.
Isaac Sim needs an NVIDIA GPU with RT cores.

## Fairness rules

Cross-engine benchmarks are trivially easy to rig by accident, so here are the
controls, stated up front:

- **Physics only**, no rendering, on both sides. Rendering is Isaac Sim's
  advantage and MuJoCo does not compete there — including it would bias this
  as badly as excluding it biases the other way.
- **Identical scene parameters**: same body count, box size, timestep, and
  step count.
- **Warm-up steps excluded** on both sides. PhysX allocates GPU buffers
  lazily; charging that to steady-state throughput would be unfair.
- **Same accuracy metric**: relative energy drift of the same undamped
  pendulum over 10 simulated seconds.

Full generator source is in the linked dataset — `bench_mujoco.py` and
`bench_isaac.py` emit an identical JSON schema on purpose.

In [ ]:
import glob, json, os

# Prefer the dataset if attached; otherwise fall back to results embedded
# below so this notebook runs standalone with no setup at all.
DATA = "/kaggle/input/mujoco-vs-isaac-benchmark"

EMBEDDED_MUJOCO = json.loads(r"""{"engine": "mujoco","version": "3.12.0","platform": "macOS-26.7-arm64-arm-64bit","machine": "arm64","accelerator": "CPU","throughput": [{"n_boxes": 1,"steps": 2000,"wall_s": 0.0071,"steps_per_s": 281548.1,"realtime_factor": 1126.19,"n_contacts_final": 4},{"n_boxes": 8,"steps": 2000,"wall_s": 0.0446,"steps_per_s": 44865.9,"realtime_factor": 179.46,"n_contacts_final": 32},{"n_boxes": 32,"steps": 2000,"wall_s": 0.1998,"steps_per_s": 10010.0,"realtime_factor": 40.04,"n_contacts_final": 128},{"n_boxes": 128,"steps": 2000,"wall_s": 0.7357,"steps_per_s": 2718.7,"realtime_factor": 10.87,"n_contacts_final": 509},{"n_boxes": 512,"steps": 2000,"wall_s": 4.8597,"steps_per_s": 411.5,"realtime_factor": 1.65,"n_contacts_final": 2287}],"accuracy": [{"dt": 0.01,"integrator": "Euler","seconds": 10.0,"rel_energy_drift_final": 0.0031773463666467205,"rel_energy_drift_max": 0.0036629493564012},{"dt": 0.01,"integrator": "RK4","seconds": 10.0,"rel_energy_drift_final": 1.18065986619641e-06,"rel_energy_drift_max": 1.5062681917967003e-06},{"dt": 0.005,"integrator": "Euler","seconds": 10.0,"rel_energy_drift_final": 0.0016362684976390192,"rel_energy_drift_max": 0.00181127442613719},{"dt": 0.005,"integrator": "RK4","seconds": 10.0,"rel_energy_drift_final": 1.4087484375721107e-07,"rel_energy_drift_max": 1.7852482235141677e-07},{"dt": 0.002,"integrator": "Euler","seconds": 10.0,"rel_energy_drift_final": 0.0006651717468267735,"rel_energy_drift_max": 0.0007196702355656947},{"dt": 0.002,"integrator": "RK4","seconds": 10.0,"rel_energy_drift_final": 8.817983090127101e-09,"rel_energy_drift_max": 1.1127338774856674e-08},{"dt": 0.001,"integrator": "Euler","seconds": 10.0,"rel_energy_drift_final": 0.0003343047013307526,"rel_energy_drift_max": 0.0003590092287424367},{"dt": 0.001,"integrator": "RK4","seconds": 10.0,"rel_energy_drift_final": 1.0952409019775168e-09,"rel_energy_drift_max": 1.37983345445304e-09}]}""")

def find(name):
    """Locate a file under /kaggle/input without assuming the mount path.

    Kaggle mounts a dataset at /kaggle/input/<slug>/, but the slug is not
    always what you expect, so search rather than hardcode.
    """
    for root in (DATA, "/kaggle/input"):
        if os.path.isdir(root):
            hits = glob.glob(os.path.join(root, "**", name), recursive=True)
            if hits:
                return hits[0]
    return None

def load(engine, embedded=None):
    p = find(f"bench_{engine}.json")
    if p:
        print(f"loaded {engine} results from {p}")
        return json.load(open(p))
    print(f"using embedded {engine} results (no bench_{engine}.json found)")
    if os.path.isdir("/kaggle/input"):
        print("  /kaggle/input contains:", os.listdir("/kaggle/input") or "(empty)")
    return embedded

mj = load("mujoco", EMBEDDED_MUJOCO)
isaac = load("isaac", None)

print("\nMuJoCo", mj["version"], "on", mj["platform"])
print("accelerator:", mj["accelerator"])

## Result 1 — Throughput

How many physics steps per second, as the scene gets more contact-heavy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def series(res):
    return ([r["n_boxes"] for r in res["throughput"]],
            [r["steps_per_s"] for r in res["throughput"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

n, sps = series(mj)
axes[0].loglog(n, sps, "o-", lw=2.5, ms=8, label=f"MuJoCo ({mj['accelerator']})")
axes[1].semilogx(n, [r["realtime_factor"] for r in mj["throughput"]],
                 "o-", lw=2.5, ms=8, label="MuJoCo")

if isaac:
    n2, sps2 = series(isaac)
    axes[0].loglog(n2, sps2, "s-", lw=2.5, ms=8,
                   label=f"Isaac Sim ({isaac['accelerator'][:20]})")
    axes[1].semilogx(n2, [r["realtime_factor"] for r in isaac["throughput"]],
                     "s-", lw=2.5, ms=8, label="Isaac Sim")

axes[0].set_xlabel("bodies in scene"); axes[0].set_ylabel("physics steps / second")
axes[0].set_title("Throughput vs scene complexity")
axes[1].axhline(1.0, color="crimson", ls="--", lw=1.5, label="real time")
axes[1].set_xlabel("bodies in scene"); axes[1].set_ylabel("realtime factor (x)")
axes[1].set_title("How much faster than wall-clock")
for a in axes: a.grid(alpha=0.3, which="both"); a.legend()
plt.tight_layout(); plt.show()

print(f"{'bodies':>8} {'steps/s':>12} {'realtime':>10} {'contacts':>10}")
for r in mj["throughput"]:
    print(f"{r['n_boxes']:>8} {r['steps_per_s']:>12,.0f} "
          f"{r['realtime_factor']:>9.2f}x {r.get('n_contacts_final','-'):>10}")

Note the shape: throughput falls off roughly with the contact count, not the
body count. At 512 boxes there are ~2300 active contacts and MuJoCo is barely
above real time on a CPU — that is the regime where GPU engines earn their
keep, and where Isaac Sim's parallel solver pulls ahead.

Below a few dozen bodies, a CPU engine on a laptop is *thousands* of times
faster than real time, and GPU dispatch overhead means the GPU engine has
nothing to offer. **This is the single most useful practical takeaway:** for
a single manipulator or a small mobile robot, CPU MuJoCo is not a compromise,
it is the correct choice.

## Result 2 — Accuracy

Now the part that speed-only benchmarks miss. An undamped pendulum conserves
energy exactly. Measure the drift and you measure integrator error directly,
with no ground-truth dataset required.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))

for integ, marker in [("Euler", "o"), ("RK4", "s")]:
    rows = [r for r in mj["accuracy"] if r["integrator"] == integ]
    if not rows: continue
    dts = [r["dt"] for r in rows]
    drift = [r["rel_energy_drift_final"] for r in rows]
    ax.loglog(dts, drift, marker + "-", lw=2.5, ms=9, label=f"MuJoCo {integ}")

if isaac and isaac.get("accuracy"):
    rows = isaac["accuracy"]
    ax.loglog([r["dt"] for r in rows],
              [r["rel_energy_drift_final"] for r in rows],
              "^-", lw=2.5, ms=9, label="Isaac Sim (PhysX TGS)")

# Reference slopes: 1st and 4th order convergence
dts = np.array([0.001, 0.01])
ax.loglog(dts, 3e-4 * (dts/0.001)**1, ":", color="gray", lw=1.5,
          label="1st order slope")
ax.loglog(dts, 1.1e-9 * (dts/0.001)**4, ":", color="darkgreen", lw=1.5,
          label="4th order slope")

ax.set_xlabel("timestep dt (s)")
ax.set_ylabel("relative energy drift after 10 s")
ax.set_title("Integrator accuracy — lower is better")
ax.grid(alpha=0.3, which="both"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Verify the convergence order empirically rather than eyeballing the slope.
for integ in ("Euler", "RK4"):
    rows = sorted([r for r in mj["accuracy"] if r["integrator"] == integ],
                  key=lambda r: -r["dt"])
    if len(rows) < 2: continue
    big, small = rows[0], rows[-1]
    ratio_dt = big["dt"] / small["dt"]
    ratio_err = big["rel_energy_drift_final"] / small["rel_energy_drift_final"]
    order = np.log(ratio_err) / np.log(ratio_dt)
    print(f"{integ:>6}: dt reduced {ratio_dt:.0f}x -> error reduced "
          f"{ratio_err:>10,.0f}x  => empirical order ~{order:.2f}")

That is the result worth the whole notebook.

**Euler comes out at ~1st order and RK4 at ~3rd–4th order**, measured, not
assumed. Which means: dropping your timestep by 10x buys you 10x less error
with Euler and roughly a thousandfold less with RK4 — at maybe 4x the cost per
step.

The practical consequence for robot learning: **if your sim is inaccurate, a
smaller timestep is usually the wrong fix.** Switching integrator is
dramatically cheaper. People burn enormous amounts of compute stepping at
1 kHz with Euler when RK4 at 100 Hz would be both faster and more accurate.

This is also why sim2real transfer fails in ways that look mysterious. A
policy trained against a 1st-order integrator has learned dynamics that
quietly gain or lose energy over long horizons, and the real robot does not.

## Result 3 — Batching, which is the actual point

Comparing "MuJoCo on CPU" to "Isaac Sim on GPU" conflates two different
variables: the **engine** and the **parallelism model**. Most of the speedup
people attribute to GPU simulators comes from the second one.

**MJX** lets us separate them. It is the same MuJoCo physics compiled through
JAX, so running it batched isolates parallelism by itself — same solver, same
model, same timestep, only the batch size changes.

In [ ]:
EMBEDDED_MJX = json.loads(r"""{"engine": "mjx","mujoco_version": "3.12.0","jax_version": "0.11.1","backend": "cpu","devices": ["cpu:0"],"platform": "macOS-26.7-arm64-arm-64bit","batch_scaling": [{"batch_size": 1,"steps_per_env": 300,"total_env_steps": 300,"compile_s": 1.221,"wall_s": 0.0424,"env_steps_per_s": 7082.3,"realtime_factor_total": 35.4,"speedup_vs_batch1": 1.0},{"batch_size": 4,"steps_per_env": 300,"total_env_steps": 1200,"compile_s": 1.326,"wall_s": 0.0495,"env_steps_per_s": 24242.9,"realtime_factor_total": 121.2,"speedup_vs_batch1": 3.42},{"batch_size": 16,"steps_per_env": 300,"total_env_steps": 4800,"compile_s": 1.163,"wall_s": 0.0548,"env_steps_per_s": 87514.3,"realtime_factor_total": 437.6,"speedup_vs_batch1": 12.36},{"batch_size": 64,"steps_per_env": 300,"total_env_steps": 19200,"compile_s": 1.596,"wall_s": 0.0803,"env_steps_per_s": 239185.2,"realtime_factor_total": 1195.9,"speedup_vs_batch1": 33.77},{"batch_size": 256,"steps_per_env": 300,"total_env_steps": 76800,"compile_s": 1.369,"wall_s": 0.0924,"env_steps_per_s": 831288.8,"realtime_factor_total": 4156.4,"speedup_vs_batch1": 117.38},{"batch_size": 1024,"steps_per_env": 300,"total_env_steps": 307200,"compile_s": 1.267,"wall_s": 0.2028,"env_steps_per_s": 1514971.6,"realtime_factor_total": 7574.9,"speedup_vs_batch1": 213.91}]}""")

mjx = EMBEDDED_MJX
_p = os.path.join(DATA, "bench_mjx.json")
if os.path.exists(_p):
    mjx = json.load(open(_p))
    print("loaded MJX results from attached dataset")

print(f"JAX backend: {mjx['backend']}  devices: {mjx['devices']}")
print(f"{'batch':>7} {'env steps/s':>14} {'speedup':>10}")
for r in mjx["batch_scaling"]:
    print(f"{r['batch_size']:>7} {r['env_steps_per_s']:>14,.0f} "
          f"{r['speedup_vs_batch1']:>9.1f}x")

In [ ]:
rows = mjx["batch_scaling"]
bs  = [r["batch_size"] for r in rows]
sp  = [r["speedup_vs_batch1"] for r in rows]
eps = [r["env_steps_per_s"] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].loglog(bs, eps, "o-", lw=2.5, ms=8, color="#2a9d8f")
axes[0].set_xlabel("parallel environments")
axes[0].set_ylabel("env steps / second")
axes[0].set_title(f"MJX throughput vs batch size ({mjx['backend'].upper()})")

axes[1].loglog(bs, sp, "o-", lw=2.5, ms=8, color="#2a9d8f", label="measured")
axes[1].loglog(bs, bs, "--", lw=1.5, color="gray", label="perfect linear scaling")
axes[1].set_xlabel("parallel environments")
axes[1].set_ylabel("speedup vs batch=1")
axes[1].set_title("How far from ideal?")
axes[1].legend()

for a in axes:
    a.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

best = rows[-1]
print(f"at batch={best['batch_size']}: {best['env_steps_per_s']:,.0f} env steps/s"
      f" = {best['realtime_factor_total']:,.0f}x realtime")
print(f"efficiency vs perfect scaling: "
      f"{100*best['speedup_vs_batch1']/best['batch_size']:.1f}%")

Two things stand out, and both are practical.

**Batching is worth more than hardware.** Going from 1 to 1024 parallel
environments bought a ~214x throughput increase *on a laptop CPU*. That is the
same mechanism Isaac Lab uses on a GPU — the GPU just starts from a much higher
base and saturates much later.

**Scaling is strongly sublinear.** 1024x more environments returned ~214x more
throughput — roughly 21% of ideal. The curve visibly bends away from the dashed
line. Every parallel simulator has a saturation point where adding environments
stops helping, and finding yours empirically is worth more than copying a
`--num_envs` value out of someone else's config file.

That has a direct cost consequence: if you are paying for a GPU and running
8192 environments because a paper did, you may be buying VRAM that returns
almost nothing. Measure the knee of your own curve.

## So which should you use?

Not a tie, but not a single winner either — the honest answer is that they
are good at different things.

**Use MuJoCo when:**
- Your scene is small (< ~50 bodies) — you get 100–1000x real time on a CPU
- You need to run anywhere: laptop, CI, Apple Silicon, a free notebook
- You are doing system identification or control, where integrator accuracy
  dominates
- You want fast iteration; startup is milliseconds, not minutes

**Use Isaac Sim when:**
- You need **photorealistic rendering** — this is not close, MuJoCo has no
  answer here, and it is the reason Isaac Sim exists
- You need **synthetic training data** with segmentation, depth, and bounding
  boxes (see notebooks 2 and 3 in this series)
- You have thousands of contacts or thousands of parallel environments
- You need USD/Omniverse interop, ROS bridges, or sensor models

**A pattern worth stealing:** many teams use both. MuJoCo for controller
development and sys-id where speed and accuracy matter, Isaac Sim for
perception data generation and final validation where fidelity matters. They
are not competitors so much as different tools that happen to both contain a
physics solver.

### Reproduce it

Both benchmark scripts are in the linked dataset and emit the same JSON
schema. The MuJoCo side runs on any machine including this notebook; the
Isaac Sim side needs an NVIDIA RT-core GPU (notebook 1 covers getting one
free).

If you run this on different hardware, **post your numbers in the comments** —
I would like to collect a table across GPUs, and a T4 vs L4 vs 4090 comparison
would be genuinely useful to everyone reading.

---

## Reproducing this

Every notebook in this series runs on free infrastructure. Nothing here needs
a paid GPU.

| Method | Free allowance | Best for |
|---|---|---|
| Kaggle | 30 GPU hr/week, 2x T4 | Running this notebook as-is |
| Lightning AI | 80 GPU hr/month, persistent disk | Heavy Isaac Sim generation |
| Google Colab | best-effort T4 | Quick smoke tests |
| NVIDIA DLI | free hosted labs | Learning the Isaac Sim GUI |

**The one gotcha worth remembering:** Isaac Sim needs **RT cores**. A T4, L4,
L40S or any RTX card is fine. An **A100 or H100 is not** — those have no RT
cores, so the RTX renderer is unsupported or unusably slow. It is the most
counterintuitive constraint in cloud robotics simulation, and it bites people
who assume the more expensive GPU must be the better one.

*Series index and full source: see the linked dataset description.*